# Aula 5,5 — Métricas de Avaliação: Precisão, Recall, MAP, nDCG e MRR (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel)*

In [1]:
# 1. Setup oficial do motor05b e inspecao do estado herdado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor05b.R")
docs <- docs_aula()
cfg  <- cfg_aula()
ix   <- montar(docs, cfg)
estado()

[1] "MOTOR_VERSAO: motor05b v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 5 | elementos: limpar, acentos, stopwords, k1, b |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao ordena..., d2 o modelo de espaco vetorial repre..., ... |
| `ix` | list | 11 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings, dl, avgdl, idf_bm25 |


## Módulo 1 — O cenário: ranking, gabarito e limiar viram rel
Alinhamento dos rankings devolvidos pelo sistema com o vetor binário de relevância `rel` a partir do limiar $\ge 2$.

In [2]:
# Módulo 1: Construcao dos vetores binarios de relevancia rel
rk_bm25 <- ranking_bm25("modelo de recuperacao", ix, cfg)
ranking <- names(rk_bm25)
grau    <- gabarito_aula()
limiar  <- 2

relevantes <- names(grau)[grau >= limiar]
rel <- as.integer(ranking %in% relevantes)

cat("Ranking BM25 (documentos ordenados):\n")
print(ranking)
cat("\nRelevantes no gabarito (grau >= 2):\n")
print(relevantes)
cat("\nVetor rel do BM25 (gabarito na ordem do ranking):\n")
print(rel)

# Checkpoint 1: rel do Cosseno
rk_cos  <- ranking_cosseno("modelo de recuperacao", ix, cfg)
rel_cos <- as.integer(names(rk_cos) %in% relevantes)
cat("\nCheckpoint 1 - Vetor rel do Cosseno:\n")
print(rel_cos)

Ranking BM25 (documentos ordenados):
[1] "d3" "d1" "d2" "d4" "d8" "d6" "d5" "d7"

Relevantes no gabarito (grau >= 2):
[1] "d2" "d3"

Vetor rel do BM25 (gabarito na ordem do ranking):
[1] 1 0 1 0 0 0 0 0

Checkpoint 1 - Vetor rel do Cosseno:
[1] 0 1 0 1 0 0 0 0


## Módulos 2 e 3 — Precisão, Recall e o Corte @k
Avaliação em pontos de corte operacionais $k$ e distinção entre frações do que foi exibido versus frações do que existe no acervo.

In [3]:
# Módulos 2 e 3: Conceitos fundamentais de Precision e Recall @k
R <- length(relevantes)  # total de relevantes no gabarito = 2

# Avaliando P@1, P@3, P@4 e R@3 no BM25
cat("BM25 com k = 1: P@1 =", sum(rel[1:1]) / 1, "| R@1 =", sum(rel[1:1]) / R, "\n")
cat("BM25 com k = 3: P@3 =", round(sum(rel[1:3]) / 3, 3), "| R@3 =", sum(rel[1:3]) / R, "\n")
cat("BM25 com k = 4: P@4 =", sum(rel[1:4]) / 4, "| R@4 =", sum(rel[1:4]) / R, "\n")

# Checkpoint 3: P@3 e R@3 no Cosseno
cat("\nCheckpoint 3 - Cosseno com k = 3:\n")
cat("P@3 =", round(sum(rel_cos[1:3]) / 3, 3), "| R@3 =", sum(rel_cos[1:3]) / R, "\n")

BM25 com k = 1: P@1 = 1 | R@1 = 0.5 
BM25 com k = 3: P@3 = 0.667 | R@3 = 1 
BM25 com k = 4: P@4 = 0.5 | R@4 = 1 

Checkpoint 3 - Cosseno com k = 3:
P@3 = 0.333 | R@3 = 0.5 


## Módulo 4 — P@k e R@k: a tabela e o dente de serra
Computação cumulativa via `cumsum()` e observação do padrão dente de serra da precisão frente aos degraus do recall.

In [4]:
# Módulo 4: Curva completa de P@k e R@k
k <- seq_along(rel)
acertos  <- cumsum(rel)
precisao <- acertos / k
recall   <- acertos / R

tabela <- round(rbind(P_at_k = precisao, R_at_k = recall), 3)
colnames(tabela) <- paste0("k=", k)
cat("Tabela de P@k e R@k para o BM25 (notar o dente de serra em k=3):\n")
print(tabela)

# Tabela do Cosseno para comparacao
acertos_cos  <- cumsum(rel_cos)
precisao_cos <- acertos_cos / k
recall_cos   <- acertos_cos / R
tab_cos <- round(rbind(P_at_k = precisao_cos, R_at_k = recall_cos), 3)
colnames(tab_cos) <- paste0("k=", k)
cat("\nTabela de P@k e R@k para o Cosseno (dentes em k=2 e k=4):\n")
print(tab_cos)

Tabela de P@k e R@k para o BM25 (notar o dente de serra em k=3):
       k=1 k=2   k=3 k=4 k=5   k=6   k=7  k=8
P_at_k 1.0 0.5 0.667 0.5 0.4 0.333 0.286 0.25
R_at_k 0.5 0.5 1.000 1.0 1.0 1.000 1.000 1.00

Tabela de P@k e R@k para o Cosseno (dentes em k=2 e k=4):
       k=1 k=2   k=3 k=4 k=5   k=6   k=7  k=8
P_at_k   0 0.5 0.333 0.5 0.4 0.333 0.286 0.25
R_at_k   0 0.5 0.500 1.0 1.0 1.000 1.000 1.00


## Módulo 5 — Average Precision (AP) e MAP
Cálculo da precisão média nas posições dos relevantes com penalização explícita por relevantes não encontrados (divisor $R$).

In [5]:
# Módulo 5: Average Precision (AP)
cat("Valores de P@k nas posicoes onde caiu relevante (rel == 1):\n")
print(round(precisao[rel == 1], 3))

ap_bm25 <- sum(precisao[rel == 1]) / R
cat("\nAP do BM25: (1.000 + 0.667) / 2 =", round(ap_bm25, 3), "\n")

# Checkpoint 5: AP do Cosseno
ap_cos <- sum(precisao_cos[rel_cos == 1]) / R
cat("Checkpoint 5 - AP do Cosseno: (0.500 + 0.500) / 2 =", round(ap_cos, 3), "\n")

Valores de P@k nas posicoes onde caiu relevante (rel == 1):
[1] 1.000 0.667

AP do BM25: (1.000 + 0.667) / 2 = 0.833 
Checkpoint 5 - AP do Cosseno: (0.500 + 0.500) / 2 = 0.5 


## Módulo 6 — Mean Reciprocal Rank (MRR)
Medição da posição do primeiro documento relevante via $1 / \text{rank}_1$ com a função `which()`.

In [6]:
# Módulo 6: Reciprocal Rank (RR) e MRR
cat("Posicoes de todos os relevantes no BM25:\n")
print(which(rel == 1))

rr_bm25 <- 1 / which(rel == 1)[1]
cat("RR do BM25 (1o relevante na posicao 1):", round(rr_bm25, 3), "\n")

# Checkpoint 6: RR do Cosseno
rr_cos <- 1 / which(rel_cos == 1)[1]
cat("\nCheckpoint 6 - RR do Cosseno (1o relevante na posicao 2):", round(rr_cos, 3), "\n")

Posicoes de todos os relevantes no BM25:
[1] 1 3
RR do BM25 (1o relevante na posicao 1): 1 

Checkpoint 6 - RR do Cosseno (1o relevante na posicao 2): 0.5 


## Módulo 7 — nDCG Binário: o desconto logarítmico
Soma descontada de ganhos via $\frac{1}{\log_2(i + 1)}$ normalizada pelo ranking ideal IDCG.

In [7]:
# Módulo 7: nDCG binario com desconto log2(i + 1)
dcg <- function(g) sum(g / log2(seq_along(g) + 1))

dcg_bm25_bin  <- dcg(rel)
idcg_bin      <- dcg(sort(rel, decreasing = TRUE))
ndcg_bm25_bin <- dcg_bm25_bin / idcg_bin

cat("DCG binario do BM25 :", round(dcg_bm25_bin, 4), "(1/1 + 1/2 = 1.5)\n")
cat("IDCG binario ideal  :", round(idcg_bin, 4), "(1/1 + 1/1.585 = 1.631)\n")
cat("nDCG binario BM25   :", round(ndcg_bm25_bin, 3), "\n")

# Checkpoint 7: nDCG binario do Cosseno
dcg_cos_bin  <- dcg(rel_cos)
ndcg_cos_bin <- dcg_cos_bin / idcg_bin
cat("\nCheckpoint 7 - nDCG binario do Cosseno:\n")
cat("DCG: ", round(dcg_cos_bin, 4), "| nDCG: ", round(ndcg_cos_bin, 3), "\n")

DCG binario do BM25 : 1.5 (1/1 + 1/2 = 1.5)
IDCG binario ideal  : 1.6309 (1/1 + 1/1.585 = 1.631)
nDCG binario BM25   : 0.92 

Checkpoint 7 - nDCG binario do Cosseno:
DCG:  1.0616 | nDCG:  0.651 


## Módulo 8 — nDCG Graduado e Síntese Comparativa
Incorporação dos graus de relevância contínuos 0, 1 e 2 e síntese comparativa das cinco métricas lado a lado.

In [8]:
# Módulo 8: nDCG graduado com graus de relevancia 0, 1 e 2
g_bm25 <- grau[ranking]
g_cos  <- grau[names(rk_cos)]

cat("Vetor de graus na ordem do BM25:\n")
print(g_bm25)

dcg_bm25_grad  <- dcg(g_bm25)
idcg_grad      <- dcg(sort(g_bm25, decreasing = TRUE))
ndcg_bm25_grad <- dcg_bm25_grad / idcg_grad

cat("\nDCG graduado do BM25 :", round(dcg_bm25_grad, 4), "\n")
cat("IDCG graduado ideal  :", round(idcg_grad, 4), "(graus 2 2 1 1 0 0 0 0)\n")
cat("nDCG graduado BM25   :", round(ndcg_bm25_grad, 3), "\n")

# Checkpoint 8: nDCG graduado do Cosseno
dcg_cos_grad  <- dcg(g_cos)
ndcg_cos_grad <- dcg_cos_grad / idcg_grad
cat("\nCheckpoint 8 - nDCG graduado do Cosseno:\n")
cat("DCG: ", round(dcg_cos_grad, 4), "| nDCG: ", round(ndcg_cos_grad, 3), "\n\n")

# Sintese das 5 metricas lado a lado
resumo_bm25 <- c(P_at_3 = precisao[[3]], AP = ap_bm25, MRR = rr_bm25, 
                 nDCG_bin = ndcg_bm25_bin, nDCG_grad = ndcg_bm25_grad)
resumo_cos  <- c(P_at_3 = precisao_cos[[3]], AP = ap_cos, MRR = rr_cos, 
                 nDCG_bin = ndcg_cos_bin, nDCG_grad = ndcg_cos_grad)

tabela_final <- round(rbind(BM25 = resumo_bm25, Cosseno = resumo_cos), 3)
cat("=== Tabela Final Comparativa das 5 Metricas ===\n")
print(tabela_final)

Vetor de graus na ordem do BM25:
d3 d1 d2 d4 d8 d6 d5 d7 
 2  1  2  0  0  1  0  0 

DCG graduado do BM25 : 3.9871 
IDCG graduado ideal  : 4.1925 (graus 2 2 1 1 0 0 0 0)
nDCG graduado BM25   : 0.951 

Checkpoint 8 - nDCG graduado do Cosseno:
DCG:  3.5101 | nDCG:  0.837 

=== Tabela Final Comparativa das 5 Metricas ===
        P_at_3    AP   MRR nDCG_bin nDCG_grad
BM25     0.667 0.833 1.000    0.920     0.951
Cosseno  0.333 0.500 0.500    0.651     0.837


## Fechamento da Sessão e Snapshot de Estado
Anexação da memória do R no consolidado.

In [9]:
# Fechamento e persistencia de estado
if (file.exists("aula05b_consolidado.md")) {
  anexar_estado("aula05b_consolidado.md")
  cat("Estado anexado a aula05b_consolidado.md com sucesso!\n")
}
cat("Sessao teorica da Aula 5,5 finalizada!\n")

Estado anexado a aula05b_consolidado.md com sucesso!
Sessao teorica da Aula 5,5 finalizada!
